# Distance to TLS and bronchi
In this notebook, we calculate set up distance axes: distance to TLS and distance to bronchi zone. Here we include the interior distance in the TLS as negative values. 

**Pinned Environment:** [`conda_envs/space2_20250604.yml`](../conda_envs/space2_20250604.yml)  

In [1]:
from pathlib import Path
import sys
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import squidpy as sq

from scipy.spatial.distance import cdist
import geopandas as gpd

import pickle

import matplotlib as mpl
mpl.rcParams['axes.titlesize'] = 24
mpl.rcParams['pdf.fonttype'] = 42

warnings.simplefilter(action='ignore', category=Warning)

## Local file info

In [2]:
sys.path.append(str(Path.cwd().resolve().parents[0]))

from config.paths import BASE_OUTDIR, INPUTS_DIR, FUNCTIONS_DIR

inputs_dir = INPUTS_DIR
structures_dir = os.path.join(BASE_OUTDIR, "structures")
out_dir = os.path.join(BASE_OUTDIR, "downstream_analysis")

plot_out_dir = os.path.join(out_dir, 'plots')

In [3]:
import importlib
if str(FUNCTIONS_DIR) not in sys.path:
    sys.path.append(str(FUNCTIONS_DIR))

import plotting_utils as pu
import de_utils as du

import distance_utils
importlib.reload(distance_utils)
from distance_utils import plot_celltype_density_2d


In [4]:
importlib.reload(du)

<module 'de_utils' from '/home/workspace/spatial_tls_manuscript/utils/de_utils.py'>

## Load adata

In [5]:
adata = sc.read_h5ad(os.path.join(out_dir,'adata_distance.h5ad'))

## Load palettes

In [6]:
with open(os.path.join(inputs_dir, 'palettes/20250519_celltype_palette_mapped.pkl'), 'rb') as f:
    celltype_palette_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, 'palettes/20250520_celltype_palette_coarse_mapped.pkl'), 'rb') as f:
    celltype_palette_coarse_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, 'palettes/20250520_celltype_palette_coarser_mapped.pkl'), 'rb') as f:
    celltype_palette_coarser_mapped = pickle.load(f)

with open(os.path.join(inputs_dir, 'palettes/20250521_zone_consol_palette_mapped.pkl'), 'rb') as f:
    zone_consol_palette_mapped = pickle.load(f)

## Calculate distance to TLS and distance to bronchi
We calculate the continuous distance to the TLS and the bronchi. 
- For the TLS, we use the polygons drawn around the TLS zone, which have been size filtered to exclude small TLS zone regions (see [`07_identify_structures.ipynb`](../initial_processing/07_identify_structures.ipynb)). Negative distances represent cells inside the TLS (interior distance from TLS edge), and positive distances represent cells outside the TLS.
- For the bronchi, we use the average distance to the nearnest n cells classified as zone bronchi. Cells in the bronchi zone are distance = 0.


In [7]:
def cell_dist_to_zone(adata, cell_type_col, zone_col, zone_label, nearest_n, dist_col):
    """
    Compute each cell's average distance to the nearest cells of a given zone.

    Cells already in `zone_label` are set to distance 0. For every other cell, the
    distance is the mean Euclidean distance (from its centroid) to its `nearest_n`
    closest cells belonging to `zone_label`, computed per cell type in `cell_type_col`.
    Results are written to `adata.obs[dist_col]`.

    Parameters
    ----------
    adata : AnnData
        AnnData with 'x_centroid'/'y_centroid' and `zone_col`/`cell_type_col` in obs.
    cell_type_col : str
        obs column whose categories the distance loop iterates over.
    zone_col : str
        obs column holding zone labels.
    zone_label : str
        The zone to measure distance to (cells in it are set to 0).
    nearest_n : int
        Number of nearest zone cells to average over.
    dist_col : str
        Name of the obs column to write distances into.

    Returns
    -------
    AnnData
        The input `adata` with `dist_col` added to `.obs`.
    """
    
    # get coordinates of other cell type (the one we want to calculate the distance to)
    adata_other_type = adata[adata.obs[zone_col] == zone_label, :]
    zone_label_coords = adata_other_type.obs[['x_centroid', 'y_centroid']].values
    print('adata_other_type shape ', adata_other_type.shape)

    # first set all cells in adata with zone_col == zone_label to 0
    adata.obs.loc[adata.obs[zone_col] == zone_label, dist_col] = 0
    # now only calculate distances for other zones
    adata_other_zones = adata[adata.obs[zone_col] != zone_label, :]
    
    # Iterate over each unique cell type
    for cell_type in adata_other_zones.obs[cell_type_col].unique():

        # Filter AnnData object for the current cell type
        adata_filtered = adata_other_zones[adata_other_zones.obs[cell_type_col] == cell_type, :]
        adata_filtered_cell_type_indices = adata_filtered.obs.index
        # print('adata_cell_type shape ', adata_filtered.shape)
    
        # Get the coordinates of the cells of the current type
        cell_type_coords = adata_filtered.obs[['x_centroid', 'y_centroid']].values
        
        # Calculate the pairwise distances between the current cell type and other cell type
        pairwise_distances = cdist(cell_type_coords, zone_label_coords, metric='euclidean')
    
        # Sort distances and select nearest n
        nearest_distances = np.sort(pairwise_distances, axis=1)[:, :nearest_n]
        
        # Calculate the average distance to the nearest n cells
        avg_distance = np.mean(nearest_distances, axis=1)

        # Store distances in adata.obs 
        
        # Assigning values to the new column in the original DataFrame, using cell type filtered adata indices
        adata.obs.loc[adata_filtered_cell_type_indices, dist_col] = avg_distance

    return adata

In [8]:
import numpy as np
import geopandas as gpd


def calculate_distances_to_regions(adata, structures_dir):
    """
    Calculate the signed distance from each cell to the nearest TLS edge.

    For every cell we compute the distance to the closest TLS boundary using a
    vectorized GeoPandas nearest-neighbor join (`sjoin_nearest`), then sign it:
      - cells OUTSIDE a TLS get the positive distance to the nearest TLS edge,
      - cells INSIDE a TLS get the NEGATIVE distance to that TLS edge (i.e. the
        interior distance, measured inward from the edge).
    Cells on/near the edge are ~0.

    This replaces the previous per-cell Shapely loop (which treated all cells
    inside a TLS as distance = 0) and is specific to the TLS. TLS geometries are
    read from `tls_gdf_{sample_label}.geojson` (the same source the radial-distance
    section uses).

    Parameters
    ----------
    adata : AnnData
        The AnnData object containing spatial coordinates of cells
        (`x_centroid`, `y_centroid` in obs) and a `sample_label` column.
    structures_dir : str
        Directory where the `tls_gdf_{sample_label}.geojson` files are stored.

    Returns
    -------
    adata : AnnData
        Updated AnnData object with:
          - obs['distance_to_tls'] : signed distance to nearest TLS edge
            (negative inside, positive outside).
          - obs['inside_tls'] : bool, equivalent to distance_to_tls < 0.
    """
    # Initialize output columns
    adata.obs['distance_to_tls'] = np.nan
    adata.obs['inside_tls'] = False

    # Process each sample separately
    for sample_label in adata.obs['sample_label'].cat.categories:
        print(f"Processing sample: {sample_label}")

        sample_mask = adata.obs['sample_label'] == sample_label
        adata_sample = adata[sample_mask, :]

        # Load TLS polygons for this sample; skip cleanly if absent/empty.
        gdf_path = os.path.join(structures_dir, f'tls_gdf_{sample_label}.geojson')
        if not os.path.exists(gdf_path):
            print(f"  no TLS geojson found, skipping: {gdf_path}")
            continue
        tls_gdf = gpd.read_file(gdf_path)
        if len(tls_gdf) == 0:
            print("  TLS gdf is empty, skipping")
            continue

        # Build a cell-point GeoDataFrame (preserve the obs index for assignment)
        pts = adata_sample.obs[['x_centroid', 'y_centroid']].copy()
        points_gdf = gpd.GeoDataFrame(
            pts,
            geometry=gpd.points_from_xy(pts.x_centroid, pts.y_centroid),
            crs=tls_gdf.crs,
        )

        # --- Unsigned distance to the nearest TLS edge (vectorized) ---
        # Join points against the TLS *boundaries* so the distance is to the edge
        # (valid for both inside and outside cells).
        boundary_gdf = tls_gdf[['geometry']].copy()
        boundary_gdf['geometry'] = boundary_gdf.geometry.boundary
        nearest = gpd.sjoin_nearest(points_gdf, boundary_gdf, distance_col='edge_dist')
        # sjoin_nearest can emit duplicate rows on exact ties; keep one per cell.
        nearest = nearest[~nearest.index.duplicated(keep='first')]
        edge_dist = nearest['edge_dist'].reindex(pts.index)

        # --- Inside flag (vectorized 'within' join against the polygons) ---
        inside = gpd.sjoin(points_gdf, tls_gdf[['geometry']], how='left', predicate='within')
        inside = inside[~inside.index.duplicated(keep='first')]
        inside_mask = inside['index_right'].reindex(pts.index).notna()

        # --- Sign it: negative inside, positive outside ---
        signed = edge_dist.copy()
        signed[inside_mask] = -signed[inside_mask]

        adata.obs.loc[pts.index, 'distance_to_tls'] = signed.values
        adata.obs.loc[pts.index, 'inside_tls'] = inside_mask.values

    return adata

In [9]:
# Calculate distance to TLS polygons
adata = calculate_distances_to_regions(adata, structures_dir)

Processing sample: HDM_day3


ERROR 1: PROJ: proj_create_from_database: Open of /home/workspace/environment/space2/share/proj failed


Processing sample: HDM_day30


In [ ]:
# Calculate distance to bronchi zone
zone_label = 'bronchi'
cell_type_col = 'label_fine'
zone_col = 'zone_consol'
dist_col = f'avg_distance_to_{zone_label}_zone'
nearest_n = 10 

start_time = time.time()

adata.obs[dist_col] = np.nan

for sample_label in adata.obs['sample_label'].cat.categories:
    
    print(sample_label)

    adata_sample = adata[adata.obs['sample_label']==sample_label, :]
    
    # run distance calculation for each sample 
    adata_sample = cell_dist_to_zone(adata=adata_sample, 
                                               cell_type_col=cell_type_col, 
                                               zone_col=zone_col, 
                                               zone_label=zone_label, 
                                               nearest_n=nearest_n, 
                                               dist_col=dist_col)
    
    # get distance df for each structure and add it back to the main adata object
    dist_df = adata_sample.obs[[dist_col]] 
    adata.obs.loc[dist_df.index, dist_col] = dist_df[dist_col]

adata.obs[dist_col] = pd.to_numeric(adata.obs[dist_col])

HDM_day3
adata_other_type shape  (16467, 480)
HDM_day30
adata_other_type shape  (22256, 480)


In [ ]:
adata

## Calculate radial distance from TLS center to surroundings
1. Create surrounding ring around each tls with specified buffer distance from tls edge
2. For cells in the tls, compute the radial distance to the tls centroid within each tls
3. For cells in the tls surrounding, compute the radial distance to the tls edge
4. Add these values back to adata
5. Create scaled distances: Within tls, scale distance to centroid by max distance to centroid for each tls
6. Create scaled distances: Within tls surrounding, scale distance to tls edge by max distance to edge (eg buffer distance) for each tls surrounding

In [ ]:
def create_surrounding_ring(geom, buffer_distance=50):
    """Create a single ring geometry around a polygon (TLS)."""
    return geom.buffer(buffer_distance).difference(geom)

# params
buffer_distance = 50

# Init
adata.obs['tls_surr_index'] = 'none'
adata.obs['tls_radial_dist'] = np.nan
adata.obs['tls_radial_dist_surr'] = np.nan

# Process each sample
for sample_label in adata.obs['sample_label'].cat.categories:

    print(f"Processing sample: {sample_label}")

    # Subset AnnData
    adata_sample = adata[adata.obs['sample_label'] == sample_label, :]

    # Load TLS geometries
    tls_gdf = gpd.read_file(os.path.join(structures_dir, f'tls_gdf_{sample_label}.geojson'))

    # Add centroid columns
    tls_gdf["tls_centroid_x"] = tls_gdf.geometry.centroid.x
    tls_gdf["tls_centroid_y"] = tls_gdf.geometry.centroid.y

    # Create surrounding ring geometry
    tls_gdf['geometry_ring'] = tls_gdf.geometry.apply(create_surrounding_ring, buffer_distance=buffer_distance)

    # --- Prepare TLS core and ring GeoDataFrames ---
    core_gdf = tls_gdf[['id', 'tls_centroid_x', 'tls_centroid_y', 'geometry']]
    ring_gdf = tls_gdf[['id', 'geometry_ring']].rename(columns={'geometry_ring': 'geometry'})
    ring_gdf['tls_surr_index'] = ring_gdf['id']
    core_gdf = gpd.GeoDataFrame(core_gdf, crs=tls_gdf.crs)
    ring_gdf = gpd.GeoDataFrame(ring_gdf, crs=tls_gdf.crs)

    # --- Build cell coordinate GeoDataFrame (keep obs index for assignment) ---
    points_df = adata_sample.obs[['x_centroid', 'y_centroid']].copy()
    points_gdf = gpd.GeoDataFrame(
        points_df,
        geometry=gpd.points_from_xy(points_df.x_centroid, points_df.y_centroid),
        crs=tls_gdf.crs
    )
    points_gdf["cell_index"] = points_df.index

    # --- Spatial join: cells in TLS cores -> radial distance to centroid ---
    core_joined = gpd.sjoin(points_gdf, core_gdf, how='left', predicate='within')
    core_joined = core_joined[~core_joined['id'].isna()]
    core_joined['tls_radial_dist'] = np.hypot(
        core_joined['x_centroid'] - core_joined['tls_centroid_x'],
        core_joined['y_centroid'] - core_joined['tls_centroid_y']
    )
    adata.obs.loc[core_joined.index, 'tls_radial_dist'] = core_joined['tls_radial_dist']

    # --- Spatial join: cells in TLS surrounding rings -> distance to TLS edge ---
    ring_joined = gpd.sjoin(points_gdf, ring_gdf, how='left', predicate='within')
    ring_joined = ring_joined[~ring_joined['tls_surr_index'].isna()]

    # Record ring membership
    adata.obs.loc[ring_joined['cell_index'], 'tls_surr_index'] = ring_joined['tls_surr_index'].astype(str).values

    # Vectorized distance to TLS edge. Ring cells lie outside the core, so a
    # point's distance to its TLS polygon equals its distance to the TLS edge.
    tls_geom_by_id = tls_gdf.set_index('id').geometry
    ring_pts = gpd.GeoSeries(ring_joined.geometry.values, crs=tls_gdf.crs)
    ring_polys = gpd.GeoSeries(ring_joined['tls_surr_index'].map(tls_geom_by_id).values, crs=tls_gdf.crs)
    edge_dist = ring_pts.distance(ring_polys, align=False)
    adata.obs.loc[ring_joined['cell_index'], 'tls_radial_dist_surr'] = edge_dist.values


# Make category dtype
adata.obs['tls_surr_index'] = adata.obs['tls_surr_index'].astype('category')

# Set surrounding ring fields to NaN for cells inside TLS
in_tls_mask = adata.obs['tls_index'].notna()
adata.obs.loc[in_tls_mask, 'tls_surr_index'] = np.nan
adata.obs.loc[in_tls_mask, 'tls_radial_dist_surr'] = np.nan


In [ ]:
# Scale radial distances by the max within each TLS.
# tls_index / tls_surr_index restart at 0 per sample, so build sample-unique
# keys to avoid pooling same-numbered TLS across samples. Masking each key with
# .where(...notna()) confines groups to cells that actually have the distance,
# so non-ring / 'none' / in-TLS cells stay NaN and don't form spurious groups.
sample = adata.obs['sample_label'].astype(str)
core_key = sample.str.cat(adata.obs['tls_index'].astype(str), sep='_').where(adata.obs['tls_radial_dist'].notna())
surr_key = sample.str.cat(adata.obs['tls_surr_index'].astype(str), sep='_').where(adata.obs['tls_radial_dist_surr'].notna())

# Scale core: distance to centroid / max distance to centroid within each TLS
adata.obs['tls_radial_dist_scaled'] = (
    adata.obs['tls_radial_dist'] / adata.obs.groupby(core_key)['tls_radial_dist'].transform('max')
)

# Scale surround: distance to edge / max distance to edge within each ring
adata.obs['tls_radial_dist_surr_scaled'] = (
    adata.obs['tls_radial_dist_surr'] / adata.obs.groupby(surr_key)['tls_radial_dist_surr'].transform('max')
)


In [ ]:
# Combine the scaled distances: [0,1] inside the TLS, [1,2] in the surrounding ring
adata.obs['tls_radial_dist_scaled_combined'] = adata.obs['tls_radial_dist_scaled']
ring_mask = adata.obs['tls_radial_dist_surr_scaled'].notna()
adata.obs.loc[ring_mask, 'tls_radial_dist_scaled_combined'] = 1 + adata.obs.loc[ring_mask, 'tls_radial_dist_surr_scaled']


In [ ]:
# Categorize TLS-associated cells by radial position (core -> outer -> surround).
# Non-TLS cells (NaN combined distance) stay NaN.
adata.obs['tls_radial_category'] = np.nan
adata.obs.loc[adata.obs['tls_radial_dist_scaled_combined'] <= 0.5, 'tls_radial_category'] = 'tls core'
adata.obs.loc[adata.obs['tls_radial_dist_scaled_combined'] > 0.5,  'tls_radial_category'] = 'tls outer'
adata.obs.loc[adata.obs['tls_radial_dist_scaled_combined'] > 1.0,  'tls_radial_category'] = 'tls surround'


In [ ]:
print('tls')
print(np.min(adata.obs['tls_radial_dist']))
print(np.max(adata.obs['tls_radial_dist']))

print('surrounding')
print(np.min(adata.obs['tls_radial_dist_surr']))
print(np.max(adata.obs['tls_radial_dist_surr']))

In [ ]:
print('tsls scaled')
print(np.min(adata.obs['tls_radial_dist_scaled']))
print(np.max(adata.obs['tls_radial_dist_scaled']))

print('surrounding scaled')
print(np.min(adata.obs['tls_radial_dist_surr_scaled']))
print(np.max(adata.obs['tls_radial_dist_surr_scaled']))

## Gate spatial regions based by distance to TLS and distance to bronchi 
Here, we leverage the concept of IMmune Allocation Plots (IMAPs) introduced in Reina-Campos et al., Nature, 2025.

In [ ]:
def assign_spatial_region(
    adata: sc.AnnData,
    gates: dict,
    x_axis: str = 'avg_distance_to_bronchi_zone',
    y_axis: str = 'distance_to_tls',
    region_col: str = 'spatial_region',
    default: str = 'unassigned',
):
    """
    Label every cell with the spatial region it falls into, based on `gates`.

    Uses the same gate definition as `plot_celltype_density_2d` so what you
    draw is what you assign. Applied to ALL cells in `adata` (no clipping),
    writing the result to `adata.obs[region_col]` as an ordered Categorical.

    A cell is in a gate if its (x_axis, y_axis) coordinates satisfy every
    bound present in that gate: x_min <= x < x_max and y_min <= y < y_max
    (missing bounds are treated as open). Gates are applied in dict order
    and the LAST matching gate wins, so list overlapping gates from most
    general to most specific. Cells matching no gate get `default`.

    Parameters
    ----------
    adata : AnnData
    gates : dict
        region name -> {'x_min','x_max','y_min','y_max'} (any subset; None = open).
    x_axis, y_axis : str
        Columns in adata.obs holding the distance coordinates.
    region_col : str
        Output column written to adata.obs.
    default : str
        Label for cells not captured by any gate.

    Returns
    -------
    adata : AnnData
        The same object, with adata.obs[region_col] added.
    """
    x = adata.obs[x_axis].values
    y = adata.obs[y_axis].values
    region = np.full(adata.n_obs, default, dtype=object)

    for name, b in gates.items():
        mask = np.ones(adata.n_obs, dtype=bool)
        if b.get('x_min') is not None:
            mask &= x >= b['x_min']
        if b.get('x_max') is not None:
            mask &= x < b['x_max']
        if b.get('y_min') is not None:
            mask &= y >= b['y_min']
        if b.get('y_max') is not None:
            mask &= y < b['y_max']
        region[mask] = name

    categories = list(gates.keys())
    if (region == default).any():
        categories = categories + [default]
    adata.obs[region_col] = pd.Categorical(region, categories=categories, ordered=True)
    return adata

In [ ]:
# x_axis = avg_distance_to_bronchi_zone, y_axis = distance_to_tls
sample_label = 'HDM_day3'
adata_plot = adata[adata.obs['sample_label'] == sample_label, :]
y_min = np.min(adata_plot.obs['distance_to_tls'])

gates = {
    'in-TLS': {'y_min':y_min, 'y_max': 1}, # in TLS
    'near-TLS':  {'y_min': 1, 'y_max': 100}, # close to TLS (any bronchi dist)
    'near-Br': {'x_max': 150, 'y_min': 150},      # close to bronchi, far from TLS
    'parenchyma':   {'x_min': 200, 'y_min': 150},      # far from both
}

In [ ]:
# Apply the gates to label every cell with the region it falls in.
adata = assign_spatial_region(
    adata,
    gates=gates,
    x_axis='avg_distance_to_bronchi_zone',
    y_axis='distance_to_tls',
    region_col='spatial_region',
)

adata.obs['spatial_region'].value_counts(dropna=False)

In [ ]:
# check cell distributions on IMAP
p = plot_celltype_density_2d(
    adata_plot,
    sample_label=[sample_label],
    # celltype='Th0',
    # celltype_col='label_fine',
    celltype='CD4 act',
    celltype_col='label_medium',  
    x_axis='avg_distance_to_bronchi_zone',
    y_axis='distance_to_tls',
    gates=gates,
    x_clip=450,
    y_clip=450,
    point_size=24,
    max_cells=5000
)
p.show()

In [ ]:
p = plot_celltype_density_2d(
    adata_plot,
    sample_label=[sample_label],
    celltype=None,
    x_axis='avg_distance_to_bronchi_zone',
    y_axis='distance_to_tls',
    gates=gates,
    x_clip=450,
    y_clip=450,
    point_size=24,
    max_cells = 5000,  
)

## Save

In [ ]:
adata.write_h5ad(os.path.join(out_dir,'adata_distance.h5ad'), compression='gzip')

In [ ]:
import session_info
print('active conda environment: ', os.path.basename(sys.prefix))
session_info.show()